### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

f:\PROJECTS\GenAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AIMessage(content='<think>\nOkay, so the user is asking why parrots can talk. Let me start by recalling what I know about parrots. They are part of the Psittaciformes order, right? I remember they have a unique beak shape and zygodactyl feet. Now, why do they talk? Well, I know they can mimic human speech, but what\'s the reason behind it?\n\nFirst, maybe it\'s related to their social structure. Parrots are social animals. In the wild, they live in flocks. Mimicking sounds could be a way to communicate with each other. So maybe talking is a form of social bonding or identifying family members. I think that\'s called vocal learning. Some other birds do it too, like mynas and some songbirds.\n\nAnother angle is that parrots learn by imitation. They hear sounds around them and repeat them. In the wild, this could help them learn predator warnings or other important calls. But how does that translate to talking to humans? Maybe when they\'re kept as pets, they mimic human speech to communi

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. Let me check what tools I have. There\'s a function called get_weather that takes a location parameter. Since the user mentioned Boston, I need to call that function with "Boston" as the location. I should make sure the parameters are correctly formatted in JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': 'sn6vesc4p', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 154, 'total_tokens': 251, 'completion_time': 0.149938063, 'completion_tokens_details': {'reasoning_tokens': 73}, 'prompt_time': 0.00615138, 'prompt_tokens_details': None, 'queue_time': 1.131032515, 'total_time': 0.156089443}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None

### Tool Execution Loops

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is currently sunny.


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check what tools I have available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call that function with "Boston" as the location. I\'ll make sure the parameters are correctly formatted as JSON and return the tool call as specified.\n', 'tool_calls': [{'id': 'zm3cmppwh', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 153, 'total_tokens': 250, 'completion_time': 0.161868983, 'completion_tokens_details': {'reasoning_tokens': 73}, 'prompt_time': 0.008119707, 'prompt_tokens_details': None, 'queue_time': 0.04683229, 'total_time': 0.16998869}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 's